# Full-Resolution Depth Visual Comparison: Standard DA-V2 vs Adapted Model

This notebook compares **inference-time depth maps** from:

- **Standard Depth Anything V2** (backbone output)
- **Your trained adapted model** (adapter output from your checkpoint)

It is intentionally **visual-only**:

- input image
- standard depth map
- adapted depth map
- absolute visual difference map

No quality metrics are computed.

Important: this notebook uses **native image resolution** (no resize/downscale) for both models.


In [ ]:
from pathlib import Path
import csv
import json
import os
import subprocess
import sys
from datetime import datetime
from glob import glob

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib import cm
from PIL import Image
import yaml

print('torch:', torch.__version__)


In [ ]:
# Resolve repository paths for Kaggle + local runs.
KAGGLE_WORKING = Path('/kaggle/working')
DEFAULT_REPO = KAGGLE_WORKING / 'EagleVision'
REPO_DIR = DEFAULT_REPO if DEFAULT_REPO.exists() else Path.cwd()

SRC_DIR = REPO_DIR / 'src'
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
os.environ['PYTHONPATH'] = f"{REPO_DIR}:{SRC_DIR}:{os.environ.get('PYTHONPATH', '')}"

from eaglevision.engine.checkpointing import load_checkpoint
from eaglevision.models.depth.depth_anything_wrapper import DepthAnythingWithAdapter
from eaglevision.models.rt_depthnvs import RoundTripDepthNVS

print('REPO_DIR:', REPO_DIR)
print('SRC_DIR:', SRC_DIR)


## Configuration

Set your adapted checkpoint path and full-resolution image globs here.

Checkpoint resolution order when `ADAPTED_CHECKPOINT=None`:

1. `outputs/<EXPERIMENT_TAG>/best_checkpoint.yaml`
2. newest `outputs/<EXPERIMENT_TAG>/checkpoints/*.pt`


In [ ]:
# -------------------- User config --------------------
EXPERIMENT_TAG = 'phase1_kaggle_best_longrun_v1'
ADAPTED_CHECKPOINT = None  # e.g., REPO_DIR / 'outputs' / EXPERIMENT_TAG / 'checkpoints' / 'epoch_100_final_step_XXXXXXX.pt'

DEPTH_MODE = 'metric'          # 'metric' or 'relative'
DEPTH_ENCODER = 'vits'         # 'vits' | 'vitb' | 'vitl'
METRIC_PROFILE = 'hypersim'    # used for metric mode
ADAPTER_HIDDEN_CHANNELS = 32
FREEZE_BACKBONE = True

# Provide high-quality / full-resolution images here.
# Examples:
# - '/kaggle/input/my-hq-images/*.jpg'
# - '/kaggle/input/my-hq-images/**/*.png'
CUSTOM_IMAGE_GLOBS = [
    '/kaggle/input/**/*.jpg',
    '/kaggle/input/**/*.png',
]

MAX_IMAGES = 12
SORT_IMAGES = True

# Full-resolution-safe inference controls.
# DA-V2 attention is quadratic in tokens, so large images must be tiled on 14-16 GB GPUs.
USE_TILED_INFERENCE = True
TILE_SIZE = 560              # 560 is divisible by 14; lower (448) if OOM persists
TILE_OVERLAP = 112           # overlap for seam reduction (also divisible by 14)
USE_AMP = True               # mixed precision for lower VRAM usage on CUDA

# Fair visual-comparison metrics (no GT depth required).
# These measure tiling artifacts + RGB/depth structural consistency for both models.
METRICS_SEAM_BAND_PX = 2
METRICS_EDGE_TOP_PERCENTILE = 80.0
METRICS_NON_EDGE_PERCENTILE = 60.0

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

OUTPUT_DIR = REPO_DIR / 'outputs' / f'fullres_depth_visual_compare_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('DEVICE:', DEVICE)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('USE_TILED_INFERENCE:', USE_TILED_INFERENCE)
print('TILE_SIZE:', TILE_SIZE, 'TILE_OVERLAP:', TILE_OVERLAP, 'USE_AMP:', USE_AMP)
print('METRICS_SEAM_BAND_PX:', METRICS_SEAM_BAND_PX)
print('METRICS_EDGE_TOP_PERCENTILE:', METRICS_EDGE_TOP_PERCENTILE)
print('METRICS_NON_EDGE_PERCENTILE:', METRICS_NON_EDGE_PERCENTILE)


In [ ]:
# Resolve DA-V2 backbone checkpoint.
if DEPTH_MODE == 'metric':
    ckpt_name = f'depth_anything_v2_metric_{METRIC_PROFILE}_{DEPTH_ENCODER}.pth'
else:
    ckpt_name = f'depth_anything_v2_{DEPTH_ENCODER}.pth'

BACKBONE_CKPT = REPO_DIR / 'baseline' / 'depth_anything_v2' / 'checkpoints' / ckpt_name

if not BACKBONE_CKPT.exists():
    print('[status] backbone checkpoint missing; attempting download via repo CLI')
    cmd = [
        sys.executable,
        '-m', 'baseline.depth_anything_v2', 'download',
        '--mode', 'all',
        '--profile', METRIC_PROFILE,
        '--encoder', DEPTH_ENCODER,
    ]
    print('$', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=REPO_DIR, text=True)
    if result.returncode != 0:
        raise RuntimeError('Failed to download DA-V2 checkpoint. Please download manually and rerun.')

if not BACKBONE_CKPT.exists():
    raise FileNotFoundError(f'Missing DA-V2 checkpoint: {BACKBONE_CKPT}')

print('BACKBONE_CKPT:', BACKBONE_CKPT)


In [ ]:
# Resolve adapted checkpoint.
run_dir = REPO_DIR / 'outputs' / EXPERIMENT_TAG

if ADAPTED_CHECKPOINT is None:
    best_yaml = run_dir / 'best_checkpoint.yaml'
    if best_yaml.exists():
        payload = yaml.safe_load(best_yaml.read_text(encoding='utf-8')) or {}
        maybe_ckpt = payload.get('checkpoint')
        if maybe_ckpt:
            ADAPTED_CHECKPOINT = Path(maybe_ckpt)

if ADAPTED_CHECKPOINT is None:
    ckpt_dir = run_dir / 'checkpoints'
    candidates = sorted(ckpt_dir.glob('*.pt')) if ckpt_dir.exists() else []
    if candidates:
        ADAPTED_CHECKPOINT = candidates[-1]

if ADAPTED_CHECKPOINT is None:
    raise FileNotFoundError(
        f'Could not auto-resolve adapted checkpoint in {run_dir}. '
        'Set ADAPTED_CHECKPOINT explicitly in the config cell.'
    )

ADAPTED_CHECKPOINT = Path(ADAPTED_CHECKPOINT)
if not ADAPTED_CHECKPOINT.exists():
    raise FileNotFoundError(f'Adapted checkpoint not found: {ADAPTED_CHECKPOINT}')

print('ADAPTED_CHECKPOINT:', ADAPTED_CHECKPOINT)


In [ ]:
# Load models for inference.
device = torch.device(DEVICE)

# Standard DA-V2 view: we use base_depth from wrapper (pure backbone output).
standard_model = DepthAnythingWithAdapter(
    mode=DEPTH_MODE,
    encoder=DEPTH_ENCODER,
    profile=METRIC_PROFILE,
    checkpoint_path=BACKBONE_CKPT,
    freeze_backbone=FREEZE_BACKBONE,
    adapter_hidden_channels=ADAPTER_HIDDEN_CHANNELS,
).to(device).eval()

# Adapted model: same backbone config + trained adapter weights from checkpoint.
adapted_depth_model = DepthAnythingWithAdapter(
    mode=DEPTH_MODE,
    encoder=DEPTH_ENCODER,
    profile=METRIC_PROFILE,
    checkpoint_path=BACKBONE_CKPT,
    freeze_backbone=FREEZE_BACKBONE,
    adapter_hidden_channels=ADAPTER_HIDDEN_CHANNELS,
).to(device)
adapted_model = RoundTripDepthNVS(adapted_depth_model).to(device).eval()
_ = load_checkpoint(ADAPTED_CHECKPOINT, adapted_model)

print('[status] models loaded')


In [ ]:
# Resolve full-resolution image list from globs.
image_paths = []
for pattern in CUSTOM_IMAGE_GLOBS:
    image_paths.extend(glob(pattern, recursive=True))

# Keep image files only, dedupe, optional sort, optional cap.
image_paths = [p for p in image_paths if Path(p).is_file()]
image_paths = list(dict.fromkeys(image_paths))
if SORT_IMAGES:
    image_paths = sorted(image_paths)
if MAX_IMAGES is not None and int(MAX_IMAGES) > 0:
    image_paths = image_paths[: int(MAX_IMAGES)]

if not image_paths:
    raise RuntimeError(
        'No images found from CUSTOM_IMAGE_GLOBS. '
        'Update the config cell with valid high-resolution image paths.'
    )

print('Resolved images:', len(image_paths))
for pth in image_paths[:10]:
    print('-', pth)


In [ ]:
def _valid_values(arr: np.ndarray):
    mask = np.isfinite(arr) & (arr > 0)
    if not mask.any():
        return np.array([], dtype=np.float32)
    return arr[mask].astype(np.float32)

def depth_to_rgb_shared(depth: np.ndarray, vmin: float, vmax: float, cmap_name: str = 'magma') -> np.ndarray:
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        norm = np.zeros_like(depth, dtype=np.float32)
    else:
        norm = (depth - vmin) / (vmax - vmin)
        norm = np.clip(norm, 0.0, 1.0)
    rgb = cm.get_cmap(cmap_name)(norm)[..., :3]
    return (rgb * 255.0).astype(np.uint8)

def _sliding_starts(size: int, tile: int, overlap: int) -> list[int]:
    if size <= tile:
        return [0]
    stride = max(1, tile - overlap)
    starts = list(range(0, size - tile + 1, stride))
    last = size - tile
    if starts[-1] != last:
        starts.append(last)
    return starts

def _blend_window(h: int, w: int, device: torch.device) -> torch.Tensor:
    # Raised-cosine weights reduce tile seams during stitching.
    wy = torch.hann_window(h, periodic=False, device=device).clamp_min(1e-3)
    wx = torch.hann_window(w, periodic=False, device=device).clamp_min(1e-3)
    return wy[:, None] * wx[None, :]

def _extract_depth(pack: dict[str, torch.Tensor], key: str) -> torch.Tensor:
    depth = pack[key]
    if depth.ndim == 4 and depth.shape[1] == 1:
        depth = depth[:, 0]
    if depth.ndim != 3:
        raise RuntimeError(f'Unexpected depth tensor shape for {key}: {tuple(depth.shape)}')
    return depth

def _infer_depth_full_or_tiled(
    model_fn,
    out_key: str,
    img_tensor: torch.Tensor,
    use_tiled: bool,
    tile_size: int,
    tile_overlap: int,
    use_amp: bool,
) -> torch.Tensor:
    device = img_tensor.device
    _, _, h, w = img_tensor.shape

    use_amp_ctx = bool(use_amp and device.type == 'cuda')

    def _run(x: torch.Tensor) -> torch.Tensor:
        if use_amp_ctx:
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                pack = model_fn(x)
        else:
            pack = model_fn(x)
        return _extract_depth(pack, out_key)

    if (not use_tiled) or (h <= tile_size and w <= tile_size):
        return _run(img_tensor)

    ys = _sliding_starts(h, tile_size, tile_overlap)
    xs = _sliding_starts(w, tile_size, tile_overlap)

    merged = torch.zeros((1, h, w), device=device, dtype=torch.float32)
    weight = torch.zeros((1, h, w), device=device, dtype=torch.float32)

    for y0 in ys:
        for x0 in xs:
            tile = img_tensor[:, :, y0:y0 + tile_size, x0:x0 + tile_size]
            depth_tile = _run(tile).float()  # [1, th, tw]
            th, tw = depth_tile.shape[-2:]

            wmap = _blend_window(th, tw, device=device).unsqueeze(0)  # [1, th, tw]
            merged[:, y0:y0 + th, x0:x0 + tw] += depth_tile * wmap
            weight[:, y0:y0 + th, x0:x0 + tw] += wmap

            del tile, depth_tile, wmap

    merged = merged / weight.clamp_min(1e-6)
    return merged

def _gray_from_rgb_uint8(rgb: np.ndarray) -> np.ndarray:
    rgb_f = rgb.astype(np.float32) / 255.0
    return 0.2989 * rgb_f[..., 0] + 0.5870 * rgb_f[..., 1] + 0.1140 * rgb_f[..., 2]

def _grad_mag(arr: np.ndarray) -> np.ndarray:
    gy, gx = np.gradient(arr.astype(np.float32))
    return np.sqrt(gx * gx + gy * gy).astype(np.float32)

def _safe_corr(a: np.ndarray, b: np.ndarray) -> float:
    av = a.reshape(-1).astype(np.float64)
    bv = b.reshape(-1).astype(np.float64)
    mask = np.isfinite(av) & np.isfinite(bv)
    if mask.sum() < 10:
        return float('nan')
    av = av[mask]
    bv = bv[mask]
    a_std = av.std()
    b_std = bv.std()
    if a_std < 1e-12 or b_std < 1e-12:
        return float('nan')
    return float(np.corrcoef(av, bv)[0, 1])

def _seam_boundaries(size: int, tile: int, overlap: int, use_tiled: bool) -> list[int]:
    if not use_tiled or size <= tile:
        return []
    starts = _sliding_starts(size, tile, overlap)
    # Boundaries at the start of every non-first tile.
    return [int(s) for s in starts[1:] if 0 < s < size]

def _seam_jump_metrics(depth: np.ndarray, x_bounds: list[int], y_bounds: list[int], band_px: int) -> tuple[float, float, int]:
    h, w = depth.shape
    diffs = []

    for x in x_bounds:
        for k in range(int(band_px)):
            xl = x - 1 - k
            xr = x + k
            if xl < 0 or xr >= w:
                continue
            diffs.append(np.abs(depth[:, xr] - depth[:, xl]))

    for y in y_bounds:
        for k in range(int(band_px)):
            yu = y - 1 - k
            yd = y + k
            if yu < 0 or yd >= h:
                continue
            diffs.append(np.abs(depth[yd, :] - depth[yu, :]))

    if not diffs:
        return float('nan'), float('nan'), 0

    concat = np.concatenate([d.reshape(-1) for d in diffs], axis=0)
    concat = concat[np.isfinite(concat)]
    if concat.size == 0:
        return float('nan'), float('nan'), 0

    return float(concat.mean()), float(np.percentile(concat, 95)), int(concat.size)

def _compute_depth_quality_metrics(
    depth: np.ndarray,
    rgb: np.ndarray,
    x_bounds: list[int],
    y_bounds: list[int],
    seam_band_px: int,
    edge_top_percentile: float,
    non_edge_percentile: float,
) -> dict[str, float]:
    eps = 1e-8
    gray = _gray_from_rgb_uint8(rgb)
    g_rgb = _grad_mag(gray)
    g_dep = _grad_mag(depth)

    seam_mean, seam_p95, seam_count = _seam_jump_metrics(depth, x_bounds, y_bounds, seam_band_px)
    global_grad_mean = float(np.mean(g_dep))
    seam_ratio = float(seam_mean / (global_grad_mean + eps)) if np.isfinite(seam_mean) else float('nan')

    edge_thr = float(np.percentile(g_rgb, edge_top_percentile))
    non_edge_thr = float(np.percentile(g_rgb, non_edge_percentile))

    edge_mask = g_rgb >= edge_thr
    non_edge_mask = g_rgb <= non_edge_thr

    edge_resp = float(np.mean(g_dep[edge_mask])) if np.any(edge_mask) else float('nan')
    non_edge_grad = float(np.mean(g_dep[non_edge_mask])) if np.any(non_edge_mask) else float('nan')
    edge_contrast = float(edge_resp / (non_edge_grad + eps)) if np.isfinite(edge_resp) and np.isfinite(non_edge_grad) else float('nan')

    edge_corr = _safe_corr(g_dep, g_rgb)

    return {
        'grad_mean': global_grad_mean,
        'seam_jump_mean': seam_mean,
        'seam_jump_p95': seam_p95,
        'seam_samples': float(seam_count),
        'seam_energy_ratio': seam_ratio,
        'rgb_edge_corr': edge_corr,
        'edge_response_mean': edge_resp,
        'non_edge_grad_mean': non_edge_grad,
        'edge_contrast_ratio': edge_contrast,
    }

panels = []
rows = []

with torch.no_grad():
    for i, image_path in enumerate(image_paths, start=1):
        print(f'[infer] {i}/{len(image_paths)} {image_path}')
        pil_img = Image.open(image_path).convert('RGB')
        input_rgb = np.array(pil_img, dtype=np.uint8)

        # Full-resolution tensor. If needed, memory-safe tiling happens inside inference helper.
        img_tensor = (
            torch.from_numpy(input_rgb)
            .float()
            .permute(2, 0, 1)
            .unsqueeze(0)
            .to(device)
            / 255.0
        )

        std_depth_t = _infer_depth_full_or_tiled(
            standard_model,
            'base_depth',
            img_tensor,
            use_tiled=USE_TILED_INFERENCE,
            tile_size=int(TILE_SIZE),
            tile_overlap=int(TILE_OVERLAP),
            use_amp=USE_AMP,
        )
        adp_depth_t = _infer_depth_full_or_tiled(
            adapted_model.depth_model,
            'adapted_depth',
            img_tensor,
            use_tiled=USE_TILED_INFERENCE,
            tile_size=int(TILE_SIZE),
            tile_overlap=int(TILE_OVERLAP),
            use_amp=USE_AMP,
        )

        std_depth = std_depth_t[0].detach().cpu().numpy()
        adp_depth = adp_depth_t[0].detach().cpu().numpy()

        # Ensure both outputs are native resolution
        h, w = input_rgb.shape[:2]
        if std_depth.shape != (h, w) or adp_depth.shape != (h, w):
            raise RuntimeError(
                f'Unexpected output shape for {image_path}: input={(h, w)} std={std_depth.shape} adapted={adp_depth.shape}'
            )

        diff = np.abs(adp_depth - std_depth)

        vals = np.concatenate([_valid_values(std_depth), _valid_values(adp_depth)])
        if vals.size == 0:
            vmin, vmax = 0.0, 1.0
        else:
            vmin, vmax = float(np.percentile(vals, 2)), float(np.percentile(vals, 98))
            if vmax <= vmin:
                vmax = vmin + 1e-6

        dvals = _valid_values(diff)
        if dvals.size == 0:
            dmin, dmax = 0.0, 1.0
        else:
            dmin, dmax = 0.0, float(np.percentile(dvals, 99))
            if dmax <= dmin:
                dmax = dmin + 1e-6

        std_rgb = depth_to_rgb_shared(std_depth, vmin, vmax, cmap_name='magma')
        adp_rgb = depth_to_rgb_shared(adp_depth, vmin, vmax, cmap_name='magma')
        diff_rgb = depth_to_rgb_shared(diff, dmin, dmax, cmap_name='inferno')

        x_bounds = _seam_boundaries(w, int(TILE_SIZE), int(TILE_OVERLAP), bool(USE_TILED_INFERENCE))
        y_bounds = _seam_boundaries(h, int(TILE_SIZE), int(TILE_OVERLAP), bool(USE_TILED_INFERENCE))

        std_m = _compute_depth_quality_metrics(
            std_depth,
            input_rgb,
            x_bounds=x_bounds,
            y_bounds=y_bounds,
            seam_band_px=int(METRICS_SEAM_BAND_PX),
            edge_top_percentile=float(METRICS_EDGE_TOP_PERCENTILE),
            non_edge_percentile=float(METRICS_NON_EDGE_PERCENTILE),
        )
        adp_m = _compute_depth_quality_metrics(
            adp_depth,
            input_rgb,
            x_bounds=x_bounds,
            y_bounds=y_bounds,
            seam_band_px=int(METRICS_SEAM_BAND_PX),
            edge_top_percentile=float(METRICS_EDGE_TOP_PERCENTILE),
            non_edge_percentile=float(METRICS_NON_EDGE_PERCENTILE),
        )

        panels.append({
            'image_path': str(image_path),
            'input_rgb': input_rgb,
            'standard_depth_rgb': std_rgb,
            'adapted_depth_rgb': adp_rgb,
            'abs_diff_rgb': diff_rgb,
            'height': h,
            'width': w,
        })

        row = {
            'sample_order': i,
            'image_path': str(image_path),
            'height': h,
            'width': w,
            'vmin': vmin,
            'vmax': vmax,
            'diff_max': dmax,
            'use_tiled_inference': bool(USE_TILED_INFERENCE),
            'tile_size': int(TILE_SIZE),
            'tile_overlap': int(TILE_OVERLAP),
            'use_amp': bool(USE_AMP),
            'seam_count_x': int(len(x_bounds)),
            'seam_count_y': int(len(y_bounds)),
        }
        for k, v in std_m.items():
            row[f'std_{k}'] = v
        for k, v in adp_m.items():
            row[f'adp_{k}'] = v
        rows.append(row)

        del img_tensor, std_depth_t, adp_depth_t, std_depth, adp_depth
        if device.type == 'cuda':
            torch.cuda.empty_cache()

print('Prepared visual panels:', len(panels))


## Fair Comparison Metrics (No GT Depth)

These metrics compare **standard** and **adapted** outputs under the same full-resolution tiling pipeline.

Lower is better:
- `seam_jump_mean`, `seam_jump_p95`, `seam_energy_ratio`, `non_edge_grad_mean`

Higher is better:
- `rgb_edge_corr`, `edge_contrast_ratio`

Note:
- This is a fair **artifact/structure** comparison, not absolute depth accuracy without ground truth.


In [ ]:
if not rows:
    raise RuntimeError('No inference rows available for metric summary.')

def _nanmean(values):
    arr = np.array(values, dtype=np.float64)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return float('nan')
    return float(arr.mean())

metric_specs = [
    ('seam_jump_mean', 'lower'),
    ('seam_jump_p95', 'lower'),
    ('seam_energy_ratio', 'lower'),
    ('non_edge_grad_mean', 'lower'),
    ('rgb_edge_corr', 'higher'),
    ('edge_contrast_ratio', 'higher'),
]

summary_rows = []
for metric_name, direction in metric_specs:
    std_key = f'std_{metric_name}'
    adp_key = f'adp_{metric_name}'

    std_vals = [r.get(std_key, float('nan')) for r in rows]
    adp_vals = [r.get(adp_key, float('nan')) for r in rows]

    std_mean = _nanmean(std_vals)
    adp_mean = _nanmean(adp_vals)

    if np.isfinite(std_mean) and np.isfinite(adp_mean):
        delta = adp_mean - std_mean
        if direction == 'lower':
            better = 'adapted' if adp_mean < std_mean else ('standard' if adp_mean > std_mean else 'tie')
            better_if = 'lower'
        else:
            better = 'adapted' if adp_mean > std_mean else ('standard' if adp_mean < std_mean else 'tie')
            better_if = 'higher'
    else:
        delta = float('nan')
        better = 'n/a'
        better_if = direction

    summary_rows.append({
        'metric': metric_name,
        'better_if': better_if,
        'standard_mean': std_mean,
        'adapted_mean': adp_mean,
        'delta_adapted_minus_standard': delta,
        'winner': better,
    })

comparison_metrics_path = OUTPUT_DIR / 'comparison_metrics_no_gt.csv'
summary_fields = [
    'metric',
    'better_if',
    'standard_mean',
    'adapted_mean',
    'delta_adapted_minus_standard',
    'winner',
]
with comparison_metrics_path.open('w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=summary_fields)
    w.writeheader()
    for row in summary_rows:
        w.writerow(row)

per_image_metrics_path = OUTPUT_DIR / 'per_image_metrics.csv'
if rows:
    per_image_fields = sorted(rows[0].keys())
    with per_image_metrics_path.open('w', encoding='utf-8', newline='') as f:
        w = csv.DictWriter(f, fieldnames=per_image_fields)
        w.writeheader()
        for row in rows:
            w.writerow(row)

print('Metric summary (means across images):')
for row in summary_rows:
    print(
        f"- {row['metric']}: std={row['standard_mean']:.6f} adp={row['adapted_mean']:.6f} "
        f"delta={row['delta_adapted_minus_standard']:.6f} winner={row['winner']} ({row['better_if']} better)"
    )

# Compact bar plot for quick visual comparison.
labels = [r['metric'] for r in summary_rows]
std_plot = [r['standard_mean'] for r in summary_rows]
adp_plot = [r['adapted_mean'] for r in summary_rows]

x = np.arange(len(labels))
width = 0.38
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.bar(x - width / 2, std_plot, width, label='Standard')
ax.bar(x + width / 2, adp_plot, width, label='Adapted')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=25, ha='right')
ax.set_title('No-GT Fair Comparison Metrics (Mean Across Images)')
ax.legend()
plt.tight_layout()

metrics_plot_path = OUTPUT_DIR / 'comparison_metrics_no_gt.png'
fig.savefig(metrics_plot_path, dpi=170, bbox_inches='tight')
plt.show()

print('Saved per-image metrics:', per_image_metrics_path)
print('Saved metric summary:', comparison_metrics_path)
print('Saved metric plot:', metrics_plot_path)


In [ ]:
# Render side-by-side visual comparison.
num = len(panels)
if num == 0:
    raise RuntimeError('No panels generated.')

fig, axes = plt.subplots(num, 4, figsize=(16, max(3.4 * num, 4)), squeeze=False)
headers = ['Input RGB (Full Res)', 'Standard DA-V2 Depth', 'Adapted Depth', 'Absolute Difference']

for c, h in enumerate(headers):
    axes[0, c].set_title(h)

for r, p in enumerate(panels):
    axes[r, 0].imshow(p['input_rgb'])
    axes[r, 1].imshow(p['standard_depth_rgb'])
    axes[r, 2].imshow(p['adapted_depth_rgb'])
    axes[r, 3].imshow(p['abs_diff_rgb'])

    label = Path(p['image_path']).name
    axes[r, 0].set_ylabel(f"{label}\n{p['width']}x{p['height']}", rotation=0, labelpad=50, va='center')

    for c in range(4):
        axes[r, c].axis('off')

plt.tight_layout()

panel_path = OUTPUT_DIR / 'fullres_depth_visual_comparison.png'
fig.savefig(panel_path, dpi=180, bbox_inches='tight')
plt.show()

print('Saved panel:', panel_path)


In [ ]:
# Save per-image outputs + manifest for downstream use.
standard_dir = OUTPUT_DIR / 'standard_depth'
adapted_dir = OUTPUT_DIR / 'adapted_depth'
diff_dir = OUTPUT_DIR / 'abs_diff'
input_dir = OUTPUT_DIR / 'inputs'
for d in [standard_dir, adapted_dir, diff_dir, input_dir]:
    d.mkdir(parents=True, exist_ok=True)

manifest_path = OUTPUT_DIR / 'inference_manifest.csv'

# Include all metric keys from rows for reproducibility.
row_fields = sorted(rows[0].keys()) if rows else []
fieldnames = [
    'sample_order', 'image_path', 'height', 'width',
    'input_path', 'standard_depth_path', 'adapted_depth_path', 'abs_diff_path',
    'adapted_checkpoint', 'backbone_checkpoint', 'mode', 'encoder', 'profile'
]
for key in row_fields:
    if key not in fieldnames:
        fieldnames.append(key)

with manifest_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

    for i, (p, r) in enumerate(zip(panels, rows), start=1):
        stem = f"sample_{i:03d}_{Path(p['image_path']).stem}"

        input_path = input_dir / f'{stem}_input.png'
        standard_path = standard_dir / f'{stem}_standard.png'
        adapted_path = adapted_dir / f'{stem}_adapted.png'
        diff_path = diff_dir / f'{stem}_abs_diff.png'

        Image.fromarray(p['input_rgb']).save(input_path)
        Image.fromarray(p['standard_depth_rgb']).save(standard_path)
        Image.fromarray(p['adapted_depth_rgb']).save(adapted_path)
        Image.fromarray(p['abs_diff_rgb']).save(diff_path)

        out_row = {
            'sample_order': i,
            'image_path': p['image_path'],
            'height': p['height'],
            'width': p['width'],
            'input_path': str(input_path),
            'standard_depth_path': str(standard_path),
            'adapted_depth_path': str(adapted_path),
            'abs_diff_path': str(diff_path),
            'adapted_checkpoint': str(ADAPTED_CHECKPOINT),
            'backbone_checkpoint': str(BACKBONE_CKPT),
            'mode': DEPTH_MODE,
            'encoder': DEPTH_ENCODER,
            'profile': METRIC_PROFILE,
        }
        out_row.update(r)
        writer.writerow(out_row)

print('Saved manifest:', manifest_path)
print('Output directory:', OUTPUT_DIR)


## Done

This notebook now compares depth maps on **full-resolution custom images** (no resizing), for both:

- standard DA-V2 (backbone output)
- your adapted checkpoint (adapter output)

For future datasets, just update `CUSTOM_IMAGE_GLOBS` in the config cell.
